In [ ]:
!pip install -q transformers accelerate rwkv torch tqdm pandas scipy requests matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.0/410.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 90.8 MB/s eta 0:00:00


In [ ]:
import os
os.makedirs('results/rwkv', exist_ok=True)

# Required Libraries

In [ ]:
import json
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import time
import requests
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

# Dataset

In [ ]:
def download_task_data(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to download data: {response.status_code}")

In [ ]:
task_url = "https://raw.githubusercontent.com/google/BIG-bench/main/bigbench/benchmark_tasks/logical_deduction/five_objects/task.json"

# Download the data
print("Downloading logical deductions task data...")
task_data = download_task_data(task_url)

## Basic Dataset Information

In [ ]:
print(f"Dataset name: {task_data['name']}")
print(f"Description: {task_data['description']}")
print(f"Number of examples: {len(task_data['examples'])}")

Dataset name: five_objects
Description: A five-object logical deduction task which requires deducing the order of a sequence of objects
Number of examples: 500


## Task Prefix

In [ ]:
task_prefix = task_data.get('task_prefix', '')
print(f"Task prefix: {task_prefix}")

Task prefix: The following paragraphs each describe a set of five objects arranged in a fixed order. The statements are logically consistent within each paragraph.




In [ ]:
print("\nExample input:")
print(task_data['examples'][0]['input'])
print("\nExample target scores:")
for option, score in task_data['examples'][0]['target_scores'].items():
    print(f"  {option}: {score}")


Example input:
On a shelf, there are five books: a gray book, a red book, a purple book, a blue book, and a black book. The red book is to the right of the gray book. The black book is to the left of the blue book. The blue book is to the left of the gray book. The purple book is the second from the right.

Example target scores:
  The gray book is the leftmost.: 0
  The red book is the leftmost.: 0
  The purple book is the leftmost.: 0
  The blue book is the leftmost.: 0
  The black book is the leftmost.: 1


In [ ]:
max_samples = min(200, len(task_data['examples']))
examples = task_data['examples'][:max_samples]
print(f"\nUsing {len(examples)} examples for benchmarking")


Using 200 examples for benchmarking


# RWKV Model

In [ ]:
# Load the RWKV model using Hugging Face Transformers (unquantized)
try:
    print("Loading RWKV-4 Raven 14B model using Hugging Face Transformers (unquantized)...")

    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained("RWKV/rwkv-raven-14b")

    # Load the model without quantization (use float16 to save memory)
    model = AutoModelForCausalLM.from_pretrained(
        "RWKV/rwkv-raven-14b",
        device_map="auto",
        torch_dtype=torch.float16,  # Use float16 instead of 4-bit quantization
        trust_remote_code=True,     # Important for RWKV model
    )

    print("Model loaded successfully")

    # Test the model with a simple prompt
    prompt = "Hello, my name is"
    print(f"\nTesting model with prompt: '{prompt}'")

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(inputs.input_ids, max_new_tokens=5)
    test_completion = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"Model completion: '{test_completion}'")

except Exception as e:
    print(f"Error loading model: {e}")
    raise

Loading RWKV-4 Raven 14B model using Hugging Face Transformers (unquantized)...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/264 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

pytorch_model-00030-of-00030.bin:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Model loaded successfully

Testing model with prompt: 'Hello, my name is'
Model completion: 'Hello, my name is [insert name] and'


In [ ]:
test_prompt = "Bob: What is 2+2?\n\nAlice:"
print(f"\nTesting model with prompt: '{test_prompt}'")

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(inputs.input_ids, max_new_tokens=5)
test_completion = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Model completion: '{test_completion}'")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.



Testing model with prompt: 'Bob: What is 2+2?

Alice:'
Model completion: 'Bob: What is 2+2?

Alice: 2 + 2 = 4'


# Utility Functions

In [ ]:
def format_logical_deduction_prompt(example):
    """
    Format a logical deduction task example into a prompt for the RWKV Raven model.
    Uses the Bob/Alice chat format which works best for Raven models.
    """
    # Extract input and options
    input_text = example['input']
    options = list(example['target_scores'].keys())

    # Format using the Bob/Alice chat format
    formatted_prompt = "### Instruction: " + "Answer this puzzle please. " + input_text + "\n\nChoose the correct answer from the following options:\n"

    # Add multiple choice options
    for i, option in enumerate(options):
        letter = chr(65 + i)  # A, B, C, D, E
        formatted_prompt += f"{letter}) {option}\n"

    # Add instruction to only output the letter of the answer
    formatted_prompt += "\nPlease answer with just the letter of the correct option (A, B, C, D, or E). Do not repeat the options or give explanations. Just give the letter of the option you think is correct.\n\n### Response:"

    return formatted_prompt, options

In [ ]:
def generate_response(prompt, max_new_tokens=20):
    """
    Generate a response from the RWKV model.
    """
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                inputs.input_ids,
                max_new_tokens=max_new_tokens,
                temperature=0.01,  # Very low temperature for more deterministic output
                do_sample=True,    # Sampling with very low temperature
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.pad_token_id
            )

        # Extract only the generated part (remove the prompt)
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = full_output[len(prompt):].strip()

        return response
    except Exception as e:
        print(f"Generation error: {e}")
        return f"ERROR: {str(e)}"

In [ ]:
def extract_answer_choice(response, num_options=5):
    """
    Extract the answer choice (A, B, C, D, E) from the model's response.
    Returns the letter or None if no valid answer is found.
    """
    response = response.strip().upper()

    # Valid options (adjust based on the number of choices)
    valid_options = [chr(65 + i) for i in range(num_options)]  # A, B, C, D, E

    # First check if the response is simply a valid option letter
    if response in valid_options:
        return response

    # Check if the response starts with a valid option
    for option in valid_options:
        if response.startswith(option) or response.startswith(option + ".") or response.startswith(option + ")"):
            return option

    # Check if the response contains a clear option marker
    for option in valid_options:
        if f"OPTION {option}" in response or f"ANSWER {option}" in response or f"ANSWER: {option}" in response:
            return option

    # Look for the first valid option letter in the response
    for char in response:
        if char in valid_options:
            return char

    # If no clear answer choice, return None
    return None

In [ ]:
def get_correct_option(example):
    """
    Get the correct option (letter) from the example's target scores.
    """
    options = list(example['target_scores'].keys())
    scores = list(example['target_scores'].values())
    correct_index = scores.index(1)  # Assume score of 1 indicates the correct answer
    return chr(65 + correct_index)  # Convert to letter (A, B, C, D, E)

# Testing with a Small Sample

In [ ]:
# Test with a few examples
print("Testing with a small sample...")
test_size = 3
test_results = []

for i, example in enumerate(examples[:test_size]):
    # Format prompt and get options
    prompt, options = format_logical_deduction_prompt(example)
    print(f"\nTest {i+1}:")
    print(f"Prompt: {prompt}")

    # Generate response
    response = generate_response(prompt)
    print(f"\nResponse: {response}")

    # Extract answer choice
    answer_choice = extract_answer_choice(response, len(options))
    print(f"\nExtracted answer: {answer_choice}")

    # Get correct answer
    correct_answer = get_correct_option(example)
    print(f"Correct answer: {correct_answer}")

    # Check if the answer is correct
    is_correct = answer_choice == correct_answer if answer_choice else False
    print(f"Correct: {is_correct}")

    # Add to test results
    test_results.append({
        'example_id': i,
        'prompt': prompt,
        'response': response,
        'extracted_answer': answer_choice,
        'correct_answer': correct_answer,
        'is_correct': is_correct,
        'is_ambiguous': answer_choice is None
    })

# Convert to DataFrame and check results
test_df = pd.DataFrame(test_results)
print("\nTest results:")
print(test_df[['example_id', 'extracted_answer', 'correct_answer', 'is_correct', 'is_ambiguous']])

Testing with a small sample...

Test 1:
Prompt: ### Instruction: Answer this puzzle please. On a shelf, there are five books: a gray book, a red book, a purple book, a blue book, and a black book. The red book is to the right of the gray book. The black book is to the left of the blue book. The blue book is to the left of the gray book. The purple book is the second from the right.

Choose the correct answer from the following options:
A) The gray book is the leftmost.
B) The red book is the leftmost.
C) The purple book is the leftmost.
D) The blue book is the leftmost.
E) The black book is the leftmost.

Please answer with just the letter of the correct option (A, B, C, D, or E). Do not repeat the options or give explanations. Just give the letter of the option you think is correct.

### Response:

Response: Option A) The gray book is the leftmost.

Extracted answer: A
Correct answer: E
Correct: False

Test 2:
Prompt: ### Instruction: Answer this puzzle please. On a shelf, there are f

# Full Evaluation

In [ ]:
def evaluate_logical_deduction(examples, save_every=10, save_prefix="rwkv_logical_deduction"):
    """
    Evaluate the RWKV model on logical deduction examples.

    Args:
        examples: List of task examples
        save_every: Save intermediate results every n samples
        save_prefix: Prefix for saved files

    Returns:
        DataFrame containing the evaluation results
    """
    results = []
    errors = 0

    start_time = time.time()

    for idx, example in enumerate(tqdm(examples)):
        try:
            # Format prompt and get options
            prompt, options = format_logical_deduction_prompt(example)

            # Generate response
            response = generate_response(prompt)

            # Extract answer choice
            answer_choice = extract_answer_choice(response, len(options))

            # Skip if no answer can be extracted
            if answer_choice is None:
                print(f"Warning: Could not extract answer for example {idx}")

            # Get correct answer
            correct_answer = get_correct_option(example)

            # Check if the answer is correct
            is_correct = answer_choice == correct_answer if answer_choice else False

            # Store result
            result = {
                'example_id': idx,
                'input_text': example['input'],
                'options': ', '.join(options),  # Join as string for CSV storage
                'model_response': response,
                'extracted_answer': answer_choice,
                'correct_answer': correct_answer,
                'is_correct': is_correct,
                'is_ambiguous': answer_choice is None
            }

            results.append(result)

            # Save intermediate results
            if (idx + 1) % save_every == 0:
                save_path = f'results/rwkv/{save_prefix}_results_intermediate_{idx+1}.csv'
                pd.DataFrame(results).to_csv(save_path, index=False)

                elapsed_time = time.time() - start_time
                avg_time_per_sample = elapsed_time / (idx + 1)
                estimated_total_time = avg_time_per_sample * len(examples)
                remaining_time = estimated_total_time - elapsed_time

                print(f"Processed {idx+1}/{len(examples)} examples")
                print(f"Elapsed time: {elapsed_time/60:.2f} minutes")
                print(f"Estimated time remaining: {remaining_time/60:.2f} minutes")
                print(f"Errors so far: {errors}")

                # Clean memory periodically
                gc.collect()
                torch.cuda.empty_cache()

        except Exception as e:
            errors += 1
            print(f"Error processing example {idx}: {e}")
            # Continue with the next example
            continue

    if errors > 0:
        print(f"Completed with {errors} errors out of {len(examples)} examples.")

    return pd.DataFrame(results)

In [ ]:
print("\nRunning full evaluation...")
start_time = time.time()

# Calculate how many examples to evaluate based on available resources
# You can adjust this number based on the small batch performance
full_eval_size = min(200, len(examples))

results_df = evaluate_logical_deduction(examples[:full_eval_size], save_every=10)

# Calculate total time
total_time = time.time() - start_time
print(f"Total evaluation time: {total_time/60:.2f} minutes")

# Save the full results
results_df.to_csv('results/rwkv/logical_deduction_full_results.csv', index=False)


Running full evaluation...


  0%|          | 0/200 [00:00<?, ?it/s]

Processed 10/200 examples
Elapsed time: 0.61 minutes
Estimated time remaining: 11.56 minutes
Errors so far: 0
Processed 20/200 examples
Elapsed time: 1.23 minutes
Estimated time remaining: 11.10 minutes
Errors so far: 0
Processed 30/200 examples
Elapsed time: 1.85 minutes
Estimated time remaining: 10.47 minutes
Errors so far: 0
Processed 40/200 examples
Elapsed time: 2.46 minutes
Estimated time remaining: 9.85 minutes
Errors so far: 0
Processed 50/200 examples
Elapsed time: 3.07 minutes
Estimated time remaining: 9.22 minutes
Errors so far: 0
Processed 60/200 examples
Elapsed time: 3.69 minutes
Estimated time remaining: 8.62 minutes
Errors so far: 0
Processed 70/200 examples
Elapsed time: 4.30 minutes
Estimated time remaining: 7.99 minutes
Errors so far: 0
Processed 80/200 examples
Elapsed time: 4.91 minutes
Estimated time remaining: 7.37 minutes
Errors so far: 0
Processed 90/200 examples
Elapsed time: 5.52 minutes
Estimated time remaining: 6.75 minutes
Errors so far: 0
Processed 100/20

# Metrics and Visualization

In [ ]:
# Calculate overall metrics
total_examples = len(results_df)
correct_examples = results_df['is_correct'].sum()
incorrect_examples = (~results_df['is_correct'] & ~results_df['is_ambiguous']).sum()
ambiguous_examples = results_df['is_ambiguous'].sum()

accuracy = correct_examples / total_examples
error_rate = incorrect_examples / total_examples
ambiguity_rate = ambiguous_examples / total_examples

print(f"\n===== OVERALL EVALUATION METRICS =====")
print(f"Model: RWKV-4 Raven 14B")
print(f"Dataset: BigBench Logical Deduction (Five Objects)")
print(f"Number of examples: {total_examples}")
print(f"\nAccuracy: {accuracy:.4f}")
print(f"Error Rate: {error_rate:.4f}")
print(f"Ambiguity Rate: {ambiguity_rate:.4f}")

# Calculate performance by answer position
answer_position_metrics = results_df.groupby('correct_answer').agg({
    'is_correct': 'mean',
    'example_id': 'count'
}).rename(columns={'is_correct': 'accuracy', 'example_id': 'count'})

print("\n===== PERFORMANCE BY CORRECT ANSWER POSITION =====")
print(answer_position_metrics)

# Create confusion matrix if possible
if not results_df['extracted_answer'].isna().all():
    # Filter out None values for confusion matrix
    confusion_df = results_df.dropna(subset=['extracted_answer'])

    if len(confusion_df) > 0:
        # Create a cross-tabulation matrix
        selection_matrix = pd.crosstab(
            confusion_df['extracted_answer'],
            confusion_df['correct_answer'],
            normalize='columns'  # Normalize by correct answer
        )
        print("\n===== SELECTED VS. CORRECT ANSWER ANALYSIS =====")
        print(selection_matrix)
    else:
        print("\nNo valid answers extracted for analysis.")
else:
    print("\nNo valid answers extracted for analysis.")


===== OVERALL EVALUATION METRICS =====
Model: RWKV-4 Raven 14B
Dataset: BigBench Logical Deduction (Five Objects)
Number of examples: 200

Accuracy: 0.2100
Error Rate: 0.7900
Ambiguity Rate: 0.0000

===== PERFORMANCE BY CORRECT ANSWER POSITION =====
                accuracy  count
correct_answer                 
A                  0.475     40
B                  0.000     40
C                  0.275     40
D                  0.300     40
E                  0.000     40

===== SELECTED VS. CORRECT ANSWER ANALYSIS =====
correct_answer        A      B      C     D      E
extracted_answer                                  
A                 0.475  0.425  0.475  0.35  0.425
C                 0.275  0.325  0.275  0.35  0.325
D                 0.250  0.250  0.250  0.30  0.250


In [ ]:
# Set the style for visualizations
plt.style.use('ggplot')
sns.set(font_scale=1.2)

# Create directory for visualizations
os.makedirs('results/rwkv/visualizations', exist_ok=True)

# 1. Overall Accuracy Pie Chart
fig1, ax1 = plt.subplots(figsize=(10, 8))
labels = ['Correct', 'Incorrect', 'Ambiguous']
sizes = [correct_examples, incorrect_examples, ambiguous_examples]
colors = ['#5cb85c', '#d9534f', '#f0ad4e']

ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title('RWKV-4 Raven 14B: Overall Results', fontsize=16)
plt.savefig('results/rwkv/visualizations/overall_results_pie.png', dpi=300)
plt.close()

# 2. Accuracy by Answer Position
if len(answer_position_metrics) > 0:
    fig2, ax2 = plt.subplots(figsize=(12, 8))
    sns.barplot(x=answer_position_metrics.index, y='accuracy', data=answer_position_metrics, ax=ax2)
    ax2.set_title('RWKV-4 Raven 14B: Accuracy by Correct Answer Position', fontsize=16)
    ax2.set_xlabel('Correct Answer')
    ax2.set_ylabel('Accuracy')
    ax2.set_ylim(0, 1)

    # Add count labels
    for i, (idx, row) in enumerate(answer_position_metrics.iterrows()):
        ax2.text(i, row['accuracy'] + 0.02, f"n={int(row['count'])}", ha='center')
        ax2.text(i, row['accuracy'] - 0.05, f"{row['accuracy']:.2f}", ha='center', color='white', fontweight='bold')

    plt.savefig('results/rwkv/visualizations/accuracy_by_position.png', dpi=300)
    plt.close()

# 3. Confusion Matrix - Selected vs. Correct Answers
if 'selection_matrix' in locals() and not selection_matrix.empty:
    fig3, ax3 = plt.subplots(figsize=(12, 10))
    sns.heatmap(selection_matrix, annot=True, fmt='.2f', cmap='Blues', ax=ax3)
    ax3.set_title('RWKV-4 Raven 14B: Confusion Matrix', fontsize=16)
    ax3.set_xlabel('Correct Answer')
    ax3.set_ylabel('Selected Answer')
    plt.savefig('results/rwkv/visualizations/confusion_matrix.png', dpi=300)
    plt.close()

# 4. Model's Answer Distribution
fig4, ax4 = plt.subplots(figsize=(12, 8))
answer_counts = results_df['extracted_answer'].fillna('None').value_counts().sort_index()

sns.barplot(x=answer_counts.index, y=answer_counts.values, ax=ax4)
ax4.set_title('RWKV-4 Raven 14B: Answer Distribution', fontsize=16)
ax4.set_xlabel('Selected Answer')
ax4.set_ylabel('Count')

# Add percentage labels
for i, count in enumerate(answer_counts.values):
    percentage = count / total_examples * 100
    ax4.text(i, count + 2, f"{percentage:.1f}%", ha='center')

plt.savefig('results/rwkv/visualizations/answer_distribution.png', dpi=300)
plt.close()

# 5. Learning Curve - Accuracy over Time
fig5, ax5 = plt.subplots(figsize=(14, 8))
# Calculate rolling accuracy
window_size = min(20, len(results_df) // 10) if len(results_df) >= 10 else 2
rolling_acc = results_df['is_correct'].rolling(window=window_size).mean()

ax5.plot(range(len(results_df)), rolling_acc, 'b-')
ax5.set_title(f'RWKV-4 Raven 14B: Rolling Accuracy (Window Size: {window_size})', fontsize=16)
ax5.set_xlabel('Example Index')
ax5.set_ylabel('Accuracy')
ax5.set_ylim(0, 1)
ax5.grid(True)
plt.savefig('results/rwkv/visualizations/learning_curve.png', dpi=300)
plt.close()

print("\nVisualizations saved to results/rwkv/visualizations/ directory")


Visualizations saved to results/rwkv/visualizations/ directory
